In [11]:
import pandas as pd
import numpy as np
import altair as alt

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold

### Add any other import statements you need below

In [ ]:
# 1. Click the folder icon in the left sidebar and drag your file in.
#    Files uploaded this way are DELETED when the session ends, so you'd re-upload every time.
# 2. Read it with pandas:
#
# df = pd.read_csv("my_data.csv")
# df = pd.read_excel("my_data.xlsx", sheet_name=0)    # 0 = first tab; or the tab's name
# df = pd.read_csv("my_data.tsv", sep="\t")           # tab-separated
#
# A file that is already online (a raw GitHub link, as in Lab 5) works the same way:
# df = pd.read_csv("https://raw.githubusercontent.com/<user>/<repo>/main/my_data.csv")
#
# To keep files between sessions, put them in Google Drive and mount it.
# (If a groupmate shared the folder with you, first right-click it in Drive > Organize > Add shortcut to My Drive.)
# from google.colab import drive
# drive.mount("/content/drive")
# df = pd.read_csv("/content/drive/MyDrive/<folder>/my_data.csv")

In [29]:
pd.read_excel(r'ds-government-spending/MonthlyReceiptsOutlaysDeficitSurplus.xls').head(5)


,Period,Receipts,Outlays,Deficit/Surplus (-),Borrowing from the Public,Reduction of Operating Cash,By Other Means
0,1980-10-31 00:00:00,38923,55843,16921,NaN,NaN,NaN
1,1980-11-30 00:00:00,39175,48083,8907,NaN,NaN,NaN
2,1980-12-31 00:00:00,48904,50869,1965,NaN,NaN,NaN
3,1981-01-31 00:00:00,51147,63339,12192,NaN,NaN,NaN
4,1981-02-28 00:00:00,38129,53749,15620,NaN,NaN,NaN


In [28]:
pd.read_excel(r'ds-government-spending/cpi-u-202512.xlsx', header=4).head(5)

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Dec.\n2024,Jan.\n2025,Feb.\n2025,Mar.\n2025,Apr.\n2025,May\n2025,Jun.\n2025,...,Oct.\n2025-\nNov.\n2025.1,Nov.\n2025-\nDec.\n2025.1,Seasonally adjusted effect on All Items\nNov. 2025-\nDec. 2025(1),"Standard error, median price change(2)",Largest (L) or Smallest (S) seasonally adjusted change since:(3),Largest (L) or Smallest (S) seasonally adjusted change since:(3).1,Unadjusted effect on All Items\nDec. 2024-\nDec. 2025(1),"Standard error, median price change(2).1",Largest (L) or Smallest (S) unadjusted change since:(3),Largest (L) or Smallest (S) unadjusted change since:(3).1
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Date,Percent change,NaN,NaN,Date,Percent change
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.0,All items,100,315.605,317.671,319.082,319.799,320.795,321.465,322.561,...,–,0.3,–,0.05,–,–,–,0.11,–,–
3,1.0,Food,13.657,333.566,335.517,336.274,337.751,337.747,338.598,339.498,...,–,0.7,0.097,0.07,L-Oct. 2022,0.7,0.42,0.13,L-Sep. 2025,3.1
4,2.0,Food at home,7.973,308.38,310.936,311.284,312.815,311.84,312.491,313.028,...,–,0.7,0.058,0.12,L-Aug. 2022,0.9,0.19,0.18,L-Sep. 2025,2.7


In [ ]:
file = "ds-government-spending/cpi-u-202512.xlsx"

months = pd.date_range(
    "2025-01-01", periods=12, freq="MS"
).strftime("%b %Y").tolist()

df = pd.read_excel(
    file,
    sheet_name="US",
    header=None,
    skiprows=7,
    usecols="A:B,E:P"
)

df.columns = ["Level", "Category", *months]

df = df[pd.to_numeric(df["Level"], errors="coerce").notna()]
df = df.drop(columns="Level")


df.head(5)

,Category,Jan 2025,Feb 2025,Mar 2025,Apr 2025,May 2025,Jun 2025,Jul 2025,Aug 2025,Sep 2025,Oct 2025,Nov 2025,Dec 2025
0,All items,317.671,319.082,319.799,320.795,321.465,322.561,323.048,323.976,324.8,–,324.122,324.054
1,Food,335.517,336.274,337.751,337.747,338.598,339.498,340.036,341.295,342.405,–,341.721,343.795
2,Food at home,310.936,311.284,312.815,311.84,312.491,313.028,313.263,314.608,316.042,–,313.954,315.664
3,Cereals and bakery products,355.964,357.627,358.45,355.976,360.325,360.04,360.048,359.74,361.894,–,361.255,361.984
4,Cereals and cereal products,282.3,285.624,287.383,282.614,287.38,285.075,283.665,282.729,283.727,–,287.875,286.9


In [ ]:
df = pd.read_excel(file, sheet_name="US", header=None)

eggs = pd.DataFrame({
    "Month": df.iloc[4, 3:16].str.replace("\n", " "),
    "Egg CPI": pd.to_numeric(df.iloc[57, 3:16], errors="coerce")
})

eggs = eggs.reset_index(drop=True)

alt.Chart(eggs).mark_line(point=True).encode(
    x=alt.X("Month:O", sort=None),
    y=alt.Y("Egg CPI:Q"),
    tooltip=["Month", "Egg CPI"]
).properties(
    title="Egg CPI 2024 - 2025",
)

alt.Chart(...)

In [44]:
# @markdown Run this cell to download this notebook as a webpage, `_NOTEBOOK.html`.

import google, json, nbformat

# Get the current notebook and write it to _NOTEBOOK.ipynb
raw_notebook = google.colab._message.blocking_request("get_ipynb",
                                                      timeout_sec=30)["ipynb"]
with open("_NOTEBOOK.ipynb", "w", encoding="utf-8") as ipynb_file:
  ipynb_file.write(json.dumps(raw_notebook))

# Use nbconvert to convert .ipynb to .html.
!jupyter nbconvert --to html --log-level WARN _NOTEBOOK.ipynb

# Download the .html file.
google.colab.files.download("_NOTEBOOK.html")

ModuleNotFoundError: No module named 'google'